# Bitcoin VNMAX · render do documentário — BR-c04

Baixa o pacote **BR-c04** do release público de `Changzaoo/vnmax-render`, renderiza e **envia cada capítulo pronto direto para a máquina de casa** (receptor pelo túnel; o endereço atual fica em `tunel.txt` do repositório). Não usa Google Drive nem nenhuma senha.

Use **Ambiente de execução › Executar tudo**. Se a sessão cair, rode tudo de novo: o que já chegou em casa é pulado (o capítulo que estava no meio recomeça).

In [ ]:
# ── parâmetros ──
ISO = "BR-c04"            # nome do pacote no release (ex.: BR, US, AR, BR-c02)
WORKERS = 0           # 0 = núcleos - 1 (mínimo 1)
CAPITULOS = ""        # "" = todos os do pacote; ou "2,5"
ORIGEM = "https://github.com/Changzaoo/vnmax-render/releases/download/pacotes/"
TUNEL = "https://raw.githubusercontent.com/Changzaoo/vnmax-render/main/tunel.txt"   # a URL atual do receptor
BASE = "/content/vnmax-render"   # disco local da sessão (quadros temporários)

In [ ]:
# ── o receptor de casa está no ar? (não segue sem ele) ──
import json, os, re, subprocess, time, urllib.request
def _get(u):
    return urllib.request.urlopen(urllib.request.Request(u, headers={"User-Agent": "vnmax-render/1"}), timeout=30).read().decode()
base = _get(TUNEL + "?t=%d" % time.time()).strip().splitlines()[0]
r = json.loads(_get(base + "/ping"))
print("receptor de casa:", "OK" if r.get("ok") else r, "| espaço livre lá:", r.get("livre_gb"), "GB")
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
# ── scripts e pacote, do release público ──
!mkdir -p /content/remoto "$BASE/pacotes"
!wget -q -O /content/remoto/bootstrap.sh "$ORIGEM"bootstrap.sh && wget -q -O /content/remoto/render_remoto.py "$ORIGEM"render_remoto.py
PACOTE = os.path.join(BASE, "pacotes", ISO + ".tar.gz")
if not os.path.exists(PACOTE):
    !wget -q -O "$PACOTE" "$ORIGEM$ISO".tar.gz
print(PACOTE, round(os.path.getsize(PACOTE) / 1048576, 1), "MB")

In [ ]:
# ── instala node 22, Chrome headless, ffmpeg e hyperframes (idempotente; ~3-5 min na 1ª vez) ──
!VNMAX_BASE="$BASE" bash /content/remoto/bootstrap.sh 2>&1 | tail -25

In [ ]:
# ── render: cada capítulo pronto é enviado para casa na hora ──
cmd = ["python3", "-u", "/content/remoto/render_remoto.py", PACOTE, "--base", BASE, "--enviar", TUNEL]
if WORKERS: cmd += ["--workers", str(WORKERS)]
if CAPITULOS: cmd += ["--capitulos", CAPITULOS]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in proc.stdout:
    print(re.sub(r"/up/[^/\s]+", "/up/…", linha), end="")
print("código de saída:", proc.wait())

In [ ]:
# ── onde parou (o pronto.json também foi enviado para casa) ──
import glob
for p in glob.glob(os.path.join(BASE, "saida", "*", "pronto.json")):
    print(p); print(open(p).read())